# Single-Gemini Baseline - FaithfulMed (Google 2D)

Owners: Joao and Vera | September milestone, Task #5

**Why this notebook is the most important one so far.** Every claim the project makes in
December is a comparison against this number. "The multi-agent pipeline improves
faithfulness by 15%" is meaningless unless the thing it improves on is measured, frozen,
and reproducible. This is that thing: one well-prompted Gemini call, no agents.

**What it does**

1. Loads the frozen MedAESQA set and refuses to run if the file changed.
2. Builds 40 examples. Source text is the expert answer with citations stripped.
3. Sends each to Gemini once, at temperature 0.
4. Scores readability, refusals, and a numeric hallucination probe.
5. Writes `baseline_gemini_results.csv` plus a run manifest.

**Set size.** MedAESQA has 40 questions, not the 50 on the task board. We use all 40
rather than padding with unlabeled examples from another corpus, because these 40 already
carry human faithfulness labels. In October the Verifier gets calibrated against those
same labels, on the same examples. Mixing in unlabeled text would break that.

**What this baseline does and does not measure.** Readability is measured properly.
Faithfulness is not, yet. There is no human judgment and no LLM judge here, so we report
a numeric fidelity probe as a floor and leave the real faithfulness score to the Verifier
work in October. Saying "faithfulness = X%" off this notebook would be wrong.

## 0. Setup

In [ ]:
!pip install -q google-genai textstat pandas matplotlib seaborn

In [ ]:
import os, json, re, time, hashlib, statistics
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import textstat

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 100)

### API key

Free key from https://aistudio.google.com/apikey . No billing, no credit card.

**Never paste the key into a cell.** It would be committed to a public repo. In Colab use
the key icon in the left sidebar to add a secret named `GEMINI_API_KEY`, then run the cell
below. Locally, `export GEMINI_API_KEY=...` before starting Jupyter.

**Rate limits are shared across the team.** The free tier is limited per minute and per
day, so coordinate before a full run. The call wrapper below backs off and retries on 429.

In [ ]:
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    print("key loaded from Colab secrets")
except Exception:
    if os.environ.get("GEMINI_API_KEY"):
        print("key found in environment")
    else:
        print("NO KEY. Add GEMINI_API_KEY as a Colab secret or export it locally.")

## 1. The frozen gold set

The advisor gave explicit guidance: version the evaluation set and stop editing it.
Silently changing it makes every before-and-after number meaningless.

So this notebook checks the file's SHA-256 and **refuses to run** if it does not match.
That is deliberate. A loud failure now is better than a quiet wrong number in December.

In [ ]:
GOLD = Path("data/medaesqa_v1.json")
if not GOLD.exists():
    GOLD = Path("medaesqa_v1.json")   # Colab: upload next to the notebook

EXPECTED_SHA = "a062742c1678b4379121ad030a06bda574afd376f331df41575938dd9849c383"

if not GOLD.exists():
    raise FileNotFoundError(
        "medaesqa_v1.json not found. Download from https://osf.io/ydbzq "
        "(direct link: https://osf.io/download/9d3qs/) and place it here."
    )

digest = hashlib.sha256(GOLD.read_bytes()).hexdigest()
print("sha256:", digest)
if digest != EXPECTED_SHA:
    raise ValueError(f"Gold set changed.\n  expected {EXPECTED_SHA}\n  got      {digest}")
print("hash matches the frozen set")

raw = json.loads(GOLD.read_text())
items = raw if isinstance(raw, list) else list(raw.values())
print(f"{len(items)} questions loaded")

## 2. Build the 40 examples

Source text is the **expert-curated answer**, which is the most faithful text in the
dataset and therefore a fair thing to ask a model to simplify without losing facts.

Citation markers like `[28646811]` are stripped. They are noise for a patient, and if we
left them in, the model would spend effort removing them and we would be measuring
formatting cleanup rather than simplification.

In [ ]:
def strip_citations(text):
    """Remove [12345678] markers, then fix the spaces they leave behind."""
    text = re.sub(r"\[\s*\d{6,9}(?:\s*,\s*\d{6,9})*\s*\]", "", text or "")
    text = re.sub(r"\s+([.,;:!?])", r"\1", text)
    return re.sub(r"\s+", " ", text).strip()

examples = []
for it in items:
    source = strip_citations(it["expert_curated_answer"])
    if len(source.split()) < 20:
        continue
    examples.append({
        "question_id": it["question_id"],
        "question": it["question"],
        "source": source,
        "specialty": (it.get("question_frame") or {}).get("Specialty"),
    })

print(f"{len(examples)} examples built")
print()
print("BEFORE:", items[0]["expert_curated_answer"][:150])
print()
print("AFTER :", examples[0]["source"][:150])

## 3. The baseline prompt

This is the comparison point, so the prompt has to be a **fair** single-agent attempt, not
a weak strawman. If we hobble the baseline, the multi-agent improvement in December is
manufactured rather than real.

So this prompt asks for everything the five-agent pipeline is supposed to deliver: plain
language, no added facts, no dropped facts, citations removed. One call, no agents.

In [ ]:
PROMPT = """You are helping a patient understand medical information.

Rewrite the SOURCE below so a patient can understand it.

Rules:
- Write at a 6th to 8th grade reading level. Short sentences. Common words.
- Use ONLY facts that appear in the SOURCE. Do not add anything.
- Do not leave out any medical fact that is in the SOURCE.
- Remove citation markers like [12345678].
- If the SOURCE says something is uncertain, keep that uncertainty.
- Write plain prose. No headings, no bullet points, no preamble.

SOURCE:
{source}

Patient-friendly version:"""

print(PROMPT.format(source="<the expert answer goes here>"))

## 4. The model call

**Temperature 0.** A baseline that changes between runs cannot be a baseline. This makes
the comparison reproducible, and it is also the setting we should use for the agents so
the ablation is apples to apples.

The wrapper retries on rate limits with exponential backoff. The free tier is shared
across six people, so a 429 is an expected event, not a crash.

In [ ]:
from google import genai
from google.genai import types

MODEL = "gemini-3.5-flash"
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])


def generate(source, model=MODEL, temperature=0.0, max_retries=5):
    prompt = PROMPT.format(source=source)
    delay = 2.0
    for attempt in range(max_retries):
        t0 = time.time()
        try:
            resp = client.models.generate_content(
                model=model, contents=prompt,
                config=types.GenerateContentConfig(temperature=temperature))
            return {"text": (resp.text or "").strip(),
                    "latency_s": round(time.time() - t0, 2),
                    "error": None, "attempts": attempt + 1}
        except Exception as e:
            msg = str(e)
            transient = any(k in msg.lower() for k in
                            ("429", "resource_exhausted", "rate", "503", "unavailable", "500"))
            if transient and attempt < max_retries - 1:
                time.sleep(delay); delay *= 2; continue
            return {"text": "", "latency_s": round(time.time() - t0, 2),
                    "error": msg[:300], "attempts": attempt + 1}


# Smoke test: one call before spending quota on all 40.
probe = generate(examples[0]["source"])
print("latency:", probe["latency_s"], "s   error:", probe["error"])
print()
print(probe["text"][:400])

## 5. Metrics

**Readability** is measured properly: Flesch-Kincaid, SMOG, jargon density.

**Faithfulness is not measured properly here, and we should not pretend otherwise.** A
real faithfulness score needs either human judgment or a calibrated LLM judge, and the
Verifier owner builds that in October against MedAESQA's human labels.

What we can measure cheaply and deterministically is **numeric fidelity**: every number
in the output should appear in the source. A number that does not is invented or mangled,
and in clinical text a wrong dose or threshold is exactly the failure that hurts a
patient. Treat it as a floor on hallucination, not a verdict.

In [ ]:
def readability(text):
    words = max(textstat.lexicon_count(text, removepunct=True), 1)
    return {"fk_grade": textstat.flesch_kincaid_grade(text),
            "smog": textstat.smog_index(text),
            "jargon_density": textstat.difficult_words(text) / words,
            "n_words": words}

NUM = re.compile(r"\b\d+(?:\.\d+)?\s?%?")

def numeric_fidelity(source, output):
    norm = lambda s: s.strip().rstrip("%").strip()
    src = {norm(x) for x in NUM.findall(source)}
    out = [x for x in NUM.findall(output)]
    unsupported = sorted(x for x in out if norm(x) not in src)
    return {"n_numbers_source": len(src),
            "unsupported_numbers": unsupported,
            "numeric_ok": len(unsupported) == 0}

def refused(text):
    if not text or len(text.split()) < 15:
        return True
    head = text.strip().lower()[:160]
    return any(t in head for t in ("i cannot", "i can\'t", "i\'m unable", "i am unable",
                                   "as an ai", "i\'m not able", "consult a healthcare"))

## 6. Run the baseline

40 calls. Pause between them to stay inside the shared free-tier limit.

**Coordinate before running this.** Six people share the quota.

In [ ]:
SLEEP_BETWEEN = 4.0   # seconds; raise this if the team is running jobs at the same time

records = []
for i, ex in enumerate(examples, 1):
    res = generate(ex["source"])
    records.append({**ex, **{"output": res["text"], "latency_s": res["latency_s"],
                             "error": res["error"], "attempts": res["attempts"]}})
    status = "ok" if not res["error"] else "ERR"
    print(f"[{i:2d}/{len(examples)}] {status}  {res['latency_s']:5.2f}s  qid={ex['question_id']}")
    time.sleep(SLEEP_BETWEEN)

errors = [r for r in records if r["error"]]
print(f"\ndone. {len(records) - len(errors)} succeeded, {len(errors)} failed")
for r in errors[:5]:
    print("  ", r["question_id"], r["error"][:120])

## 7. Score

In [ ]:
rows = []
for r in records:
    row = {"question_id": r["question_id"], "specialty": r["specialty"],
           "latency_s": r["latency_s"], "error": r["error"]}
    s, o = readability(r["source"]), (readability(r["output"]) if r["output"] else
                                      {"fk_grade": np.nan, "smog": np.nan,
                                       "jargon_density": np.nan, "n_words": 0})
    row.update({f"source_{k}": v for k, v in s.items()})
    row.update({f"output_{k}": v for k, v in o.items()})
    row["grade_drop"] = s["fk_grade"] - o["fk_grade"]
    row["meets_grade8"] = o["fk_grade"] <= 8.0 if o["fk_grade"] == o["fk_grade"] else False
    row["refused"] = refused(r["output"])
    row.update(numeric_fidelity(r["source"], r["output"] or ""))
    rows.append(row)

df = pd.DataFrame(rows)
ok = df[df.error.isna()]

print("=" * 58)
print("SINGLE-GEMINI BASELINE".center(58))
print("=" * 58)
print(f"model                 {MODEL}   temperature 0.0")
print(f"examples              {len(ok)} scored ({len(df) - len(ok)} failed)")
print()
print(f"source FK grade       {ok.source_fk_grade.mean():.2f}")
print(f"output FK grade       {ok.output_fk_grade.mean():.2f}")
print(f"grade drop            {ok.grade_drop.mean():.2f}")
print()
print(f"MEETS GRADE 8         {ok.meets_grade8.mean()*100:.1f}%    (target: 80%)")
print(f"refusal rate          {ok.refused.mean()*100:.1f}%")
print(f"numeric probe clean   {ok.numeric_ok.mean()*100:.1f}%")
print(f"median latency        {ok.latency_s.median():.2f}s")
print("=" * 58)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.4))

ax = axes[0]
ax.scatter(ok.source_fk_grade, ok.output_fk_grade, alpha=.75, s=45)
lim = [0, max(ok.source_fk_grade.max(), ok.output_fk_grade.max()) + 1]
ax.plot(lim, lim, color="grey", ls=":", lw=1, label="no change")
ax.axhline(8, color="crimson", ls="--", lw=1.6, label="grade 8 target")
ax.set_xlabel("source grade level"); ax.set_ylabel("Gemini output grade level")
ax.set_title("Did one Gemini call simplify the text?"); ax.legend(fontsize=8)

ax = axes[1]
ax.hist([ok.source_fk_grade, ok.output_fk_grade], bins=14,
        label=["source (expert answer)", "Gemini baseline"], color=["#DD8452", "#4C72B0"])
ax.axvline(8, color="crimson", ls="--", lw=1.6)
ax.set_xlabel("Flesch-Kincaid grade level"); ax.set_ylabel("examples")
ax.set_title("Before and after"); ax.legend(fontsize=8)

plt.tight_layout(); plt.show()

### Where the baseline fails

The examples it could not get to grade 8, and any number it invented. These are what the
multi-agent pipeline has to fix, so they are the most useful rows in the table.

In [ ]:
worst = ok.nlargest(8, "output_fk_grade")[
    ["question_id", "specialty", "source_fk_grade", "output_fk_grade", "meets_grade8"]]
print("HARDEST EXAMPLES FOR THE BASELINE")
print(worst.to_string(index=False))

bad_numbers = ok[~ok.numeric_ok]
print(f"\nNUMERIC PROBE FAILURES: {len(bad_numbers)}")
for _, r in bad_numbers.iterrows():
    print(f"  qid {r.question_id}: output numbers not in source -> {r.unsupported_numbers}")

if ok.refused.any():
    print(f"\nREFUSALS: {int(ok.refused.sum())}")
    print(ok.loc[ok.refused, ["question_id", "specialty"]].to_string(index=False))

## 8. Freeze the baseline

Same discipline as the gold set. This number is the denominator for every improvement
claim in December, so the run is recorded with its model, temperature, date and gold-set
hash. If any of those change, it is a different baseline and must be rerun, not compared.

In [ ]:
import datetime

df.to_csv("baseline_gemini_results.csv", index=False)

manifest = {
    "run_date": datetime.date.today().isoformat(),
    "model": MODEL,
    "temperature": 0.0,
    "n_examples": int(len(ok)),
    "gold_sha256": digest,
    "source_fk_grade_mean": round(float(ok.source_fk_grade.mean()), 3),
    "output_fk_grade_mean": round(float(ok.output_fk_grade.mean()), 3),
    "pct_meets_grade8": round(float(ok.meets_grade8.mean() * 100), 2),
    "refusal_rate_pct": round(float(ok.refused.mean() * 100), 2),
    "numeric_probe_clean_pct": round(float(ok.numeric_ok.mean() * 100), 2),
    "median_latency_s": round(float(ok.latency_s.median()), 2),
}
Path("baseline_manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))

## 9. What this tells the team

Fill in after running.

| Question | Answer |
|---|---|
| Does one Gemini call hit the 80% readability target? | |
| How far is it from grade 8 on average? | |
| Did it refuse any medical rewrite? | |
| Did it invent numbers? | |
| What does the multi-agent pipeline have to beat? | |

**How to use this number.** Every agent added in October gets measured against this same
40-example set with the same prompt discipline. The ablation table in November is
literally this row repeated with one agent switched off at a time.

**What this does not establish.** Faithfulness. The numeric probe is a floor, not a
faithfulness score. Until the Verifier is calibrated against MedAESQA's human labels in
October, nobody should quote a faithfulness percentage from this notebook.

**A caution for the December report.** The success criteria ask for a 15% absolute
improvement in faithfulness over this baseline. If the baseline turns out to be strong,
that bar is hard, and that is a real finding worth reporting rather than a failure to
hide. A well-prompted single call being hard to beat is exactly the kind of result the
research question was asked to discover.